# Hackathon ML 2026 : cette séance sera-t-elle difficile pour cette personne ?

Pipeline unique : chargement, choix du schéma de validation, features relatives à la personne,
comparaison de familles de modèles en validation croisée (macro-F1 out-of-fold), ensemble,
analyse d'erreur, soumission.

Exécution : `python hackathon.py` (script) ou `python tools/export_notebook.py` pour le notebook rendu.
Variables d'environnement : `HACK_MODELS=logreg,hgb` (modèles à évaluer), `HACK_SUBMIT=0` (pas de fichier de soumission),
`HACK_EXPLAIN=1` (importance par permutation), `HACK_DATA` / `HACK_OUT` (dossiers).

In [ ]:
from __future__ import annotations

import os
import sys
import time
from datetime import datetime
from pathlib import Path

import matplotlib

if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder

show = globals().get("display", print)

ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
DATA = Path(os.environ.get("HACK_DATA", ROOT / "data" if (ROOT / "data" / "train.csv").exists() else ROOT))
OUT = Path(os.environ.get("HACK_OUT", ROOT / "out"))
SUBS = OUT.parent / "submissions" if "HACK_OUT" in os.environ else ROOT / "submissions"
for d in (OUT / "figures", OUT / "oof", OUT / "test_proba", SUBS):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
N_SPLITS = 5
ID, TARGET = "workout_id", "intensity"
CLASS_ORDER = ["easy", "moderate", "hard"]
FEATURES_VERSION = "v1-relatif-personne"  # incrémenter à chaque changement de features (tracé dans experiments.csv)
RUN_MODELS = os.environ.get("HACK_MODELS", "logreg,knn,rf,extratrees,hgb").split(",")
N_JOBS = max(1, (os.cpu_count() or 2) - 1)

# Si la détection automatique des colonnes se trompe, forcer ici : {"USER": "user_id", "SPORT": "sport", ...}
OVERRIDE: dict[str, str | None] = {}

## 1. Données et schéma

In [ ]:
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
print("train", train.shape, "| test", test.shape, "| sample", sample.shape)
show(train.head())
print(train.dtypes.to_string())
print("\nValeurs manquantes (train) :\n", train.isna().mean().round(3)[lambda s: s > 0].to_string() or "aucune")
print("\nCible :\n", train[TARGET].value_counts(normalize=True).round(3).to_string())

In [ ]:
def find_col(df: pd.DataFrame, keys: list[str], numeric: bool | None = None, exclude: tuple = ()) -> str | None:
    """Première colonne dont le nom contient une clé (les clés sont testées par ordre de priorité)."""
    for k in keys:
        for c in df.columns:
            if c in (ID, TARGET) or c in exclude:
                continue
            if k in c.lower() and (numeric is None or pd.api.types.is_numeric_dtype(df[c]) == numeric):
                return c
    return None


USER = find_col(train, ["user", "utilisateur", "athlete", "person"])
SPORT = find_col(train, ["sport", "activity", "activite"])
GENDER = find_col(train, ["gender", "genre", "sex"])
START = find_col(train, ["start", "debut", "début", "timestamp", "date", "heure", "hour"], exclude=(USER,))
DURATION = find_col(train, ["duration", "duree", "durée", "elapsed", "moving_time"], numeric=True)
DISTANCE = find_col(train, ["distance", "dist"], numeric=True)
ELEV = find_col(train, ["elev", "ascent", "denivel", "dénivel", "climb", "altitude"], numeric=True)
SPEED_COLS = [c for c in train.columns if any(k in c.lower() for k in ("speed", "vitesse", "pace", "allure"))
              and pd.api.types.is_numeric_dtype(train[c])]
SPEED_MAX = next((c for c in SPEED_COLS if "max" in c.lower()), None)
SPEED_AVG = next((c for c in SPEED_COLS if any(k in c.lower() for k in ("avg", "mean", "moy"))), None)
for _k, _v in OVERRIDE.items():
    globals()[_k] = _v

print({"USER": USER, "SPORT": SPORT, "GENDER": GENDER, "START": START, "DURATION": DURATION,
       "DISTANCE": DISTANCE, "ELEV": ELEV, "SPEED_AVG": SPEED_AVG, "SPEED_MAX": SPEED_MAX})

## 2. Schéma de validation : la CV doit imiter le découpage train/test

Si les utilisateurs du test sont aussi dans le train, on peut apprendre « qui » est la personne
(encodage cible par utilisateur, validé en StratifiedKFold). Sinon, on valide par groupe d'utilisateurs
et l'identité de l'utilisateur n'apporte rien.

In [ ]:
user_overlap = float(test[USER].isin(set(train[USER])).mean()) if USER else 0.0
USE_USER_TE = USER is not None and user_overlap >= 0.5
if USER:
    print(f"Utilisateurs : {train[USER].nunique()} (train), {test[USER].nunique()} (test), "
          f"séances test dont l'utilisateur est connu : {user_overlap:.1%}")
    print("Séances par utilisateur (train) :", train[USER].value_counts().describe().round(1).to_dict())
if USE_USER_TE:
    CV, GROUPS = StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED), None
    print("=> StratifiedKFold + encodage cible par utilisateur (cross-fitted)")
else:
    CV = StratifiedGroupKFold(N_SPLITS, shuffle=True, random_state=SEED) if USER else StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED)
    GROUPS = train[USER] if USER else None
    print("=> validation groupée par utilisateur, pas d'encodage cible utilisateur")

## 2b. Hypothèse « difficile = tiers par personne » (prémisse mesurée, pas supposée)

Le briefing dit que l'intensité est mesurée séparément pour chaque personne et que les classes sont équilibrées.
Si les labels sont des tiers calculés par utilisateur, la part de chaque classe par utilisateur est beaucoup plus
serrée autour de 1/3 qu'un tirage aléatoire (binomial). Ce test décide si le post-traitement par quotas (6b) a un sens.

In [ ]:
PER_USER_TERCILES = False
if USER:
    _counts = pd.crosstab(train[USER], train[TARGET])
    _n = _counts.sum(1)
    _big = _n >= 9
    _prop = _counts[_big].div(_n[_big], axis=0)
    _obs_sd = float(_prop.std().mean())
    _binom_sd = float(np.sqrt((1 / 3) * (2 / 3) / _n[_big]).mean())
    PER_USER_TERCILES = _obs_sd < 0.6 * _binom_sd
    print(f"Écart-type observé de la part de chaque classe par utilisateur : {_obs_sd:.3f} "
          f"vs attendu si aléatoire : {_binom_sd:.3f} => tiers par utilisateur : {PER_USER_TERCILES}")
    show(_prop.describe().round(3))

## 3. Features : « difficile » est relatif à la personne

Une même vitesse est facile pour l'un, difficile pour l'autre. On décrit donc chaque séance par rapport
aux habitudes de son utilisateur (z-score, ratio, rang dans son historique), globalement et par sport.
Les statistiques sont calculées sur train + test sans la cible (aucune fuite de label, aucune donnée externe).

In [ ]:
def safe_div(a, b):
    return a / b.replace(0, np.nan)


def time_features(df: pd.DataFrame) -> pd.DataFrame:
    if START is None:
        return pd.DataFrame(index=df.index)
    s = df[START]
    f = pd.DataFrame(index=df.index)
    if pd.api.types.is_numeric_dtype(s) and s.max() <= 48:
        hour = s.astype(float)
        ts = None
    else:
        if pd.api.types.is_numeric_dtype(s):
            ts = pd.to_datetime(s, unit="ms" if s.max() > 1e12 else "s", errors="coerce")
        else:
            ts = pd.to_datetime(s, errors="coerce", format="mixed")
        if ts.dt.tz is not None:
            ts = ts.dt.tz_localize(None)
        hour = ts.dt.hour + ts.dt.minute / 60
        if ts.dt.normalize().nunique() > 1:  # vraie date disponible (sinon heure seule)
            f["dow"] = ts.dt.dayofweek
            f["month"] = ts.dt.month
        else:
            ts = None
    f["hour"] = hour
    f["hour_sin"], f["hour_cos"] = np.sin(2 * np.pi * hour / 24), np.cos(2 * np.pi * hour / 24)
    if ts is not None and USER:
        # Récupération : temps depuis la séance précédente du même utilisateur.
        days = (ts - ts.min()).dt.total_seconds() / 86400
        order = days.groupby(df[USER]).rank(method="first")
        tmp = pd.DataFrame({"u": df[USER], "d": days}).sort_values(["u", "d"])
        f["days_since_prev"] = tmp.groupby("u")["d"].diff().reindex(df.index)
        f["days_to_next"] = (-tmp.groupby("u")["d"].diff(-1)).reindex(df.index)
        f["session_index_user"] = order
    return f


def derived_features(df: pd.DataFrame) -> pd.DataFrame:
    f = pd.DataFrame(index=df.index)
    if DISTANCE and DURATION:
        f["speed_calc"] = safe_div(df[DISTANCE], df[DURATION])
    if ELEV and DISTANCE:
        f["elev_per_dist"] = safe_div(df[ELEV], df[DISTANCE])
    if ELEV and DURATION:
        f["elev_per_time"] = safe_div(df[ELEV], df[DURATION])
    if SPEED_MAX and SPEED_AVG:
        f["speed_max_over_avg"] = safe_div(df[SPEED_MAX], df[SPEED_AVG])
    return f


def relative_features(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    out: dict[str, pd.Series] = {}
    groupings = [([USER], "u"), ([USER, SPORT], "us"), ([SPORT], "s"), ([SPORT, GENDER], "sg")]
    for keys, tag in groupings:
        if any(k is None for k in keys):
            continue
        g = df.groupby(keys, observed=True, dropna=False)
        out[f"n_{tag}"] = g[keys[0]].transform("size")
        for c in cols:
            m, sd = g[c].transform("mean"), g[c].transform("std")
            out[f"{c}__z_{tag}"] = safe_div(df[c] - m, sd)
            out[f"{c}__ratio_{tag}"] = safe_div(df[c], m)
            if tag in ("u", "us"):
                out[f"{c}__rank_{tag}"] = g[c].rank(pct=True)
    return pd.DataFrame(out, index=df.index)


def build_features(train: pd.DataFrame, test: pd.DataFrame):
    all_df = pd.concat([train.drop(columns=[TARGET]), test], ignore_index=True)
    base = all_df.drop(columns=[c for c in (ID, START) if c], errors="ignore")
    num = [c for c in base.select_dtypes("number").columns if c != USER]
    feats = pd.concat([base[num], time_features(all_df), derived_features(all_df)], axis=1)
    rel_cols = [c for c in feats.columns if c not in ("dow", "month", "hour_sin", "hour_cos", "session_index_user")]
    feats = pd.concat([feats, relative_features(pd.concat([feats, all_df[[c for c in (USER, SPORT, GENDER) if c]]], axis=1), rel_cols)], axis=1)
    feats = feats.replace([np.inf, -np.inf], np.nan)
    cat = [c for c in base.columns if c not in num and c != USER and base[c].nunique() <= 50]
    num_cols = list(feats.columns)
    X_all = pd.concat([feats, all_df[cat + ([USER] if USER else [])]], axis=1)
    return X_all.iloc[: len(train)].reset_index(drop=True), X_all.iloc[len(train):].reset_index(drop=True), num_cols, cat


X, X_test, NUM_COLS, CAT_COLS = build_features(train, test)
y = train[TARGET].astype(str).reset_index(drop=True)
CLASSES = [c for c in CLASS_ORDER if c in set(y)] or sorted(y.unique())
print(f"{len(NUM_COLS)} features numériques, catégorielles : {CAT_COLS}, encodage utilisateur : {USE_USER_TE}")

## 4. Que veut dire « difficile » ? (information mutuelle avec la cible)

In [ ]:
_rng = np.random.default_rng(SEED)
_idx = _rng.choice(len(X), size=min(10000, len(X)), replace=False)
_Xmi = SimpleImputer(strategy="median").fit_transform(X.iloc[_idx][NUM_COLS])
mi = pd.Series(mutual_info_classif(_Xmi, y.iloc[_idx], random_state=SEED), index=NUM_COLS).sort_values(ascending=False)
print("Top 25 features par information mutuelle :\n", mi.head(25).round(4).to_string())
if SPORT:
    print("\nIntensité par sport :\n", pd.crosstab(train[SPORT], train[TARGET], normalize="index").round(2).to_string())

## 5. Familles de modèles comparées en validation croisée

Chaque modèle est un pipeline complet (imputation, encodage, modèle) refit dans chaque pli : pas de fuite.
Les probabilités out-of-fold sont sauvegardées pour l'ensemble et l'analyse d'erreur ; chaque run est tracé
dans `out/experiments.csv`.

In [ ]:
def make_preprocessor(scale: bool) -> ColumnTransformer:
    transformers = [("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                                      ("sc", StandardScaler() if scale else "passthrough")]), NUM_COLS)]
    if CAT_COLS:
        transformers.append(("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20,
                                                  sparse_output=False), CAT_COLS))
    if USE_USER_TE:
        transformers.append(("user_te", TargetEncoder(target_type="multiclass", cv=5, shuffle=True,
                                                      random_state=SEED), [USER]))
    return ColumnTransformer(transformers, remainder="drop", verbose_feature_names_out=False)


MODELS = {
    "logreg": (LogisticRegression(max_iter=3000, C=0.5), True),
    "knn": (KNeighborsClassifier(n_neighbors=31, weights="distance", n_jobs=N_JOBS), True),
    "rf": (RandomForestClassifier(n_estimators=400, min_samples_leaf=3, n_jobs=N_JOBS, random_state=SEED), False),
    "extratrees": (ExtraTreesClassifier(n_estimators=500, min_samples_leaf=2, n_jobs=N_JOBS, random_state=SEED), False),
    "hgb": (HistGradientBoostingClassifier(learning_rate=0.05, max_iter=800, max_leaf_nodes=31, l2_regularization=1.0,
                                           early_stopping=True, n_iter_no_change=40, random_state=SEED), False),
}


def make_pipeline(name: str) -> Pipeline:
    est, scale = MODELS[name]
    return Pipeline([("prep", make_preprocessor(scale)), ("clf", clone(est))])


def macro_f1(y_true, proba, weights=None) -> float:
    p = proba * (weights if weights is not None else 1)
    return f1_score(y_true, np.array(CLASSES)[p.argmax(1)], average="macro")


def run_cv(name: str, note: str = "") -> tuple[np.ndarray, np.ndarray]:
    """Probabilités out-of-fold sur le train, et prédictions du test par le modèle de chaque pli (pour les quotas)."""
    t0 = time.time()
    oof = np.zeros((len(X), len(CLASSES)))
    test_folds = np.zeros((N_SPLITS, len(X_test), len(CLASSES)))
    fold_scores = []
    for i, (tr, va) in enumerate(CV.split(X, y, GROUPS)):
        pipe = make_pipeline(name).fit(X.iloc[tr], y.iloc[tr])
        cols = [list(pipe.classes_).index(c) for c in CLASSES]
        oof[va] = pipe.predict_proba(X.iloc[va])[:, cols]
        test_folds[i] = pipe.predict_proba(X_test)[:, cols]
        fold_scores.append(macro_f1(y.iloc[va], oof[va]))
    score = macro_f1(y, oof)
    np.save(OUT / "oof" / f"{name}.npy", oof)
    np.save(OUT / "oof" / f"{name}_testfolds.npy", test_folds)
    rec = {"time": datetime.now().strftime("%H:%M"), "model": name, "features": FEATURES_VERSION,
           "cv": type(CV).__name__, "macro_f1_oof": round(score, 5), "fold_std": round(float(np.std(fold_scores)), 5),
           "seconds": round(time.time() - t0, 1), "note": note}
    log_path = OUT / "experiments.csv"
    pd.DataFrame([rec]).to_csv(log_path, mode="a", header=not log_path.exists(), index=False)
    print(f"{name:<11} macro-F1 OOF = {score:.4f} (± {np.std(fold_scores):.4f} entre plis, {rec['seconds']} s)")
    return oof, test_folds


_cv_results = {name: run_cv(name) for name in RUN_MODELS}
OOF = {n: r[0] for n, r in _cv_results.items()}
OOF_TEST = {n: r[1] for n, r in _cv_results.items()}

## 6. Ensemble : sélection gloutonne de modèles (moyenne de probabilités) et poids de classe pour le macro-F1

In [ ]:
def greedy_blend(oofs: dict[str, np.ndarray], max_steps: int = 10) -> list[str]:
    """Ajoute à chaque pas le modèle (avec remise) qui améliore le plus le macro-F1 OOF de la moyenne."""
    chosen: list[str] = []
    best = -1.0
    for _ in range(max_steps):
        cand = {n: macro_f1(y, np.mean([oofs[m] for m in chosen + [n]], axis=0)) for n in oofs}
        n, s = max(cand.items(), key=lambda kv: kv[1])
        if s <= best + 1e-4:
            break
        chosen.append(n)
        best = s
    return chosen


def tune_class_weights(proba: np.ndarray) -> np.ndarray:
    w = np.ones(len(CLASSES))
    for _ in range(3):
        for j in range(len(CLASSES)):
            best_v, best_s = w[j], macro_f1(y, proba, w)
            for v in np.linspace(0.7, 1.4, 29):
                w2 = w.copy()
                w2[j] = v
                s = macro_f1(y, proba, w2)
                if s > best_s + 1e-5:
                    best_v, best_s = v, s
            w[j] = best_v
    return w


BLEND = greedy_blend(OOF)
P_blend = np.mean([OOF[m] for m in BLEND], axis=0)
W = tune_class_weights(P_blend)
gain = macro_f1(y, P_blend, W) - macro_f1(y, P_blend)
if gain < 0.002:  # gain trop faible pour être distingué du bruit : on garde l'argmax simple
    W = np.ones(len(CLASSES))
print("Ensemble retenu :", BLEND, f"| macro-F1 OOF = {macro_f1(y, P_blend):.4f}",
      f"| poids de classe {dict(zip(CLASSES, W.round(2)))} (gain {gain:+.4f})")
print("\nClassement :\n", pd.Series({n: macro_f1(y, o) for n, o in OOF.items()}).sort_values(ascending=False).round(4).to_string())

## 6b. Post-traitement par quotas par utilisateur

Pour chaque personne, on trie ses séances à prédire par difficulté prédite (P(moderate) + 2 P(hard)) et on les
découpe pour que, sur l'ensemble de ses séances (connues + à prédire), chaque classe fasse environ un tiers.
Explicable sur papier ; adopté seulement si le macro-F1 OOF s'améliore.

In [ ]:
def user_quota_assign(proba: np.ndarray, users_unknown: np.ndarray, users_known: np.ndarray, labels_known: np.ndarray,
                      n_total_per_user: pd.Series) -> np.ndarray:
    score = proba @ np.arange(len(CLASSES))  # espérance ordinale : 0 = easy, 2 = hard
    known = pd.crosstab(pd.Series(users_known, name="u"), pd.Series(labels_known, name="y")).reindex(columns=CLASSES, fill_value=0)
    pred = np.array(CLASSES)[proba.argmax(1)].astype(object)
    for u, idx in pd.Series(np.arange(len(users_unknown))).groupby(users_unknown):
        idx = idx.to_numpy()
        target = np.full(len(CLASSES), n_total_per_user.get(u, len(idx)) / len(CLASSES))
        if u in known.index:
            target = target - known.loc[u].to_numpy()
        target = np.clip(target, 0, None)
        if target.sum() == 0:
            continue
        target = target / target.sum() * len(idx)
        q = np.floor(target).astype(int)
        q[np.argsort(-(target - q))[: len(idx) - q.sum()]] += 1  # plus forts restes
        order = idx[np.argsort(score[idx], kind="stable")]
        pred[order] = np.repeat(CLASSES, q)
    return pred


def quota_oof(proba: np.ndarray, proba_test_folds: np.ndarray) -> np.ndarray:
    """Simulation fidèle : dans chaque pli, les séances inconnues d'un utilisateur = pli de validation + ses séances de test."""
    out = np.empty(len(y), dtype=object)
    n_all = pd.concat([train[USER], test[USER]]).value_counts()
    for i, (tr, va) in enumerate(CV.split(X, y, GROUPS)):
        P_u = np.vstack([proba[va], proba_test_folds[i]])
        users_u = np.concatenate([train[USER].values[va], test[USER].values])
        out[va] = user_quota_assign(P_u, users_u, train[USER].values[tr], y.values[tr], n_all)[: len(va)]
    return out


USE_QUOTA = False
if USER and USE_USER_TE:
    q_pred = quota_oof(P_blend, np.mean([OOF_TEST[m] for m in BLEND], axis=0))
    q_score = f1_score(y, q_pred, average="macro")
    base = macro_f1(y, P_blend, W)
    USE_QUOTA = q_score > base + 0.002
    print(f"Quotas par utilisateur : macro-F1 OOF {q_score:.4f} vs {base:.4f} => adopté : {USE_QUOTA}")

## 7. Analyse d'erreur (matrice de confusion pour le Gate 2)

In [ ]:
def error_analysis(proba: np.ndarray, label: str, weights=None) -> None:
    pred = np.array(CLASSES)[(proba * (weights if weights is not None else 1)).argmax(1)]
    cm = confusion_matrix(y, pred, labels=CLASSES)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=axes[0], colorbar=False, values_format="d")
    ConfusionMatrixDisplay(cm / cm.sum(1, keepdims=True), display_labels=CLASSES).plot(ax=axes[1], colorbar=False, values_format=".2f")
    axes[0].set_title(f"{label} : effectifs (OOF)")
    axes[1].set_title("normalisée par vraie classe (rappel)")
    fig.tight_layout()
    fig.savefig(OUT / "figures" / f"confusion_{label}.png", dpi=120)
    if "ipykernel" in sys.modules:
        plt.show()
    plt.close(fig)
    print(classification_report(y, pred, labels=CLASSES, digits=3))
    off = cm.astype(float).copy()
    np.fill_diagonal(off, 0)
    i, j = np.unravel_index(off.argmax(), off.shape)
    print(f"Confusion principale : vrai '{CLASSES[i]}' prédit '{CLASSES[j]}' ({int(cm[i, j])} séances, "
          f"{cm[i, j] / cm[i].sum():.0%} des '{CLASSES[i]}'). Erreurs extrêmes easy<->hard : "
          f"{int(cm[0, -1] + cm[-1, 0])}.")
    if SPORT:
        per_sport = pd.DataFrame({"sport": train[SPORT].values, "y": y, "p": pred}).groupby("sport").apply(
            lambda d: pd.Series({"n": len(d), "macro_f1": f1_score(d.y, d.p, average="macro")}), include_groups=False)
        print("\nmacro-F1 par sport :\n", per_sport.sort_values("n", ascending=False).round(3).to_string())


error_analysis(P_blend, "blend", W)

## 8. Entraînement final et soumission

In [ ]:
def fit_predict_test(name: str) -> np.ndarray:
    pipe = make_pipeline(name).fit(X, y)
    cols = [list(pipe.classes_).index(c) for c in CLASSES]
    proba = pipe.predict_proba(X_test)[:, cols]
    np.save(OUT / "test_proba" / f"{name}.npy", proba)
    return proba


def write_submission(proba: np.ndarray, tag: str, weights=None) -> Path:
    labels = np.array(CLASSES)[(proba * (weights if weights is not None else 1)).argmax(1)]
    sub = pd.DataFrame({ID: test[ID].values, TARGET: labels}).set_index(ID).loc[sample[ID]].reset_index()
    assert len(sub) == len(sample) and sub[TARGET].notna().all() and set(sub[TARGET]) <= set(CLASSES)
    assert list(sub.columns) == list(sample.columns), (list(sub.columns), list(sample.columns))
    path = SUBS / f"sub_{datetime.now():%H%M}_{tag}.csv"
    sub.to_csv(path, index=False)
    print(f"Soumission écrite : {path}\nDistribution prédite : {sub[TARGET].value_counts(normalize=True).round(3).to_dict()}")
    return path


if os.environ.get("HACK_SUBMIT", "1") == "1":
    test_probas = {m: fit_predict_test(m) for m in dict.fromkeys(BLEND)}
    P_test = np.mean([test_probas[m] for m in BLEND], axis=0)
    tag = "blend-" + "+".join(dict.fromkeys(BLEND))
    if USE_QUOTA:
        n_all = pd.concat([train[USER], test[USER]]).value_counts()
        labels = user_quota_assign(P_test, test[USER].values, train[USER].values, y.values, n_all)
        P_test = (labels[:, None] == np.array(CLASSES)[None, :]).astype(float)
        tag += "-quota"
    write_submission(P_test, tag, None if USE_QUOTA else W)

## 9. Explication du modèle (pour la fiche papier) : importance par permutation sur un pli de validation

In [ ]:
if os.environ.get("HACK_EXPLAIN") == "1":
    best_single = max(OOF, key=lambda n: macro_f1(y, OOF[n]))
    tr, va = next(CV.split(X, y, GROUPS))
    pipe = make_pipeline(best_single).fit(X.iloc[tr], y.iloc[tr])
    va_sample = va[: min(4000, len(va))]
    imp = permutation_importance(pipe, X.iloc[va_sample], y.iloc[va_sample], scoring="f1_macro",
                                 n_repeats=3, random_state=SEED, n_jobs=N_JOBS)
    print(f"Importance par permutation ({best_single}, baisse de macro-F1) :\n",
          pd.Series(imp.importances_mean, index=X.columns).sort_values(ascending=False).head(20).round(4).to_string())